<a href="https://colab.research.google.com/github/bcarmoai/ML-and-AI/blob/main/RAG_Chapter3_Demo1_Generative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim numba

In [ ]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 124.9 MB/s  0:00:00


In [ ]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.1 MB/s  0:00:00


In [ ]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# Directory to store downloaded and extracted data
DATA_DIR = Path("./mimic_textbooks")

# Download and extract the dataset zip file
def download_and_extract_zip(url, extract_to=DATA_DIR):
    # Ensure the directory exists
    extract_to.mkdir(parents=True, exist_ok=True)

    # Download the zip file
    zip_path = extract_to / "textbooks.zip"
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)

    # Extract the zip file
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset downloaded and extracted.")

# Download and extract textbooks
dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
download_and_extract_zip(dataset_url)


88121KB [00:02, 32434.54KB/s]


Extracting dataset...
Dataset downloaded and extracted.


In [ ]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob

# Load text files
def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("F*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

# Cleaning and preprocessing function
def clean_and_tokenize(text):
    # Basic regex cleaning
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Lowercase all text
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters

    # Tokenize with gensim
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

# Spell correction
def correct_spelling(text):
    return str(TextBlob(text).correct())

# Chunk text into fixed-size chunks
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, correct, and chunk documents
documents = load_text_files(DATA_DIR / "textbooks/en")
cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
# corrected_documents = [correct_spelling(doc) for doc in cleaned_documents]
chunked_documents = []
for doc in cleaned_documents:
    chunked_documents.extend(chunk_text(doc))

print(f"Total document chunks created: {len(chunked_documents)}")


Total document chunks created: 1086


In [ ]:
# Show the first ten records of chunked_documents
for idx, chunk in enumerate(chunked_documents[:10]):
    print(f"--- Chunk #{idx} ---")
    print(chunk)
    print()

--- Chunk #0 ---
database of highyield facts the seventh edition of first aid for the usmle step ck contains revised and expanded database of clinical material that student authors and faculty have identified as high yield for boards review the facts are organized according to subject matter whether medical specialty eg cardiovascular renal or highyield topic eg ethics in medicine each subject is then divided into smaller subsections of related facts individual facts are generally presented in logical approach from basic definitions and epidemiology to historyphysical exam diagnosis and treatment lists mnemonics and tables are used when helpful in forming key associations the content is mostly useful for reviewing material already learned this section is not ideal for learning complex or highly conceptual material for the first time blackandwhite images appear throughout the text in some cases reference is made to the clinical image section at the end of section which contains fullcolo

In [ ]:
import random

# 1. Basic Statistics
num_chunks = len(chunked_documents)
chunk_lengths = [len(chunk.split()) for chunk in chunked_documents]
avg_length = sum(chunk_lengths) / num_chunks if num_chunks > 0 else 0

print("=== Dataset Exploration ===")
print(f"Total Number of Chunks: {num_chunks}")
print(f"Average Chunk Length (words): {avg_length:.2f}")
print(f"Min Chunk Length (words): {min(chunk_lengths) if chunk_lengths else 0}")
print(f"Max Chunk Length (words): {max(chunk_lengths) if chunk_lengths else 0}\n")

# 2. Show a few random samples
print("=== Random Samples ===")
sample_indices = random.sample(range(num_chunks), min(3, num_chunks))
for idx in sample_indices:
    print(f"--- Chunk #{idx} ({len(chunked_documents[idx].split())} words) ---")
    print(chunked_documents[idx])
    print()


=== Dataset Exploration ===
Total Number of Chunks: 1086
Average Chunk Length (words): 199.80
Min Chunk Length (words): 60
Max Chunk Length (words): 200

=== Random Samples ===
--- Chunk #419 (200 words) ---
necklace sign see figure seen in roughly twothirds of women with pcos gonadotropins lhfsh ratio fasting insulin levels hour urine for free cortisol adultonset cah or cushings syndrome igu polycystic ovary with prominent multiple cysts reproduced with permission from decherney ah current obstetric gynecologic diagnosis treatment th ed stamford ct appleton lange the risk of endometrial among women with pcos women who are not attempting to conceive treat with combination of ocps progestin and metformin or other agents women who are attempting to conceive clomiphene metformin is frstline treatment for ovulatory stimulation symptomspecif treatment hirsutism combination ocps are frst line antiandrogens spironolactone nasteride and metformin may also be used cardiovascular risk factors an

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Function to generate embeddings for all chunks in a batch
def get_embeddings_in_batch(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Tokenize the batch of texts
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        # Generate embeddings
        with torch.no_grad():
            outputs = model(**inputs).last_hidden_state  # [batch_size, sequence_length, hidden_size]
            batch_embeddings = torch.mean(outputs, dim=1).cpu().numpy()  # Mean pooling

        # Append batch embeddings to the list
        all_embeddings.extend(batch_embeddings)

    return np.array(all_embeddings)

# Generate embeddings for all document chunks in batches
embeddings = get_embeddings_in_batch(chunked_documents, batch_size=128)
print(f"Generated embeddings for {len(embeddings)} document chunks.")


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generated embeddings for 1086 document chunks.


In [ ]:
# Confirm the object type of the embeddings object
print("Object Type:", type(embeddings))

# Confirm the shape to see dimensions
print("Embeddings Shape:", embeddings.shape)

# Show the first ten records (rows) of the embeddings matrix
print("\nFirst ten embedding vectors (truncated for readability):")
print(embeddings[:10])

Object Type: <class 'numpy.ndarray'>
Embeddings Shape: (1086, 384)

First ten embedding vectors (truncated for readability):
[[ 0.04665871 -0.09081743  0.02870332 ... -0.05489047 -0.08965371
   0.01174803]
 [ 0.00880189 -0.07429098  0.20216475 ...  0.04984448 -0.15880945
  -0.05591156]
 [-0.0307402  -0.01238072 -0.15130347 ...  0.01495383 -0.06756122
  -0.05863642]
 ...
 [-0.09424475 -0.10233699 -0.07657287 ... -0.09492056  0.07205862
   0.10997009]
 [-0.14889103 -0.14990886 -0.07979389 ... -0.08923297  0.20623426
   0.07691798]
 [-0.14896451 -0.08121771  0.05271242 ... -0.01639283  0.17940234
   0.08729222]]


In [ ]:
import faiss
import numpy as np

# Define the dimension of embeddings
dimension = 384  # Embedding size from MiniLM model
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to NumPy array for FAISS
embedding_matrix = np.array([embedding.flatten() for embedding in embeddings]).astype('float32')

# Add embeddings to FAISS index
index.add(embedding_matrix)
print(f"Total embeddings indexed: {index.ntotal}")


Total embeddings indexed: 1086


In [ ]:
import torch
import gc
# from numba import cuda

try:
    del model # Free up the embedding model from memory if it exists
except NameError:
    pass

torch.cuda.empty_cache()  # Clear GPU memory from torch safely
gc.collect()

# We no longer need to forcefully reset the CUDA device because we removed TensorFlow
# device = cuda.get_current_device()
# device.reset() # <--- This is what broke PyTorch earlier!


43

## Retrival Method

In [ ]:
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch

# Load the tokenizer and model for retrieval on CPU
retrieval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retrieval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").cpu()

# Function to generate embeddings for a new query
def get_query_embedding(query):
    with torch.no_grad():
        inputs = retrieval_tokenizer(query, return_tensors="pt", padding=True, truncation=True)
        outputs = retrieval_model(**inputs)
        embedding = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
    return embedding

# Load FAISS index with existing embeddings
embedding_dim = 384
index = faiss.IndexFlatL2(embedding_dim)

# Function to retrieve relevant documents based on the query
def retrieve_documents(query, top_k=5):
    query_embedding = get_query_embedding(query).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = [chunked_documents[idx] for idx in indices[0]]
    return results

# Test retrieval component
sample_query = "What are the symptoms of heart failure?"
similar_documents = retrieve_documents(sample_query)
print("Retrieved documents:", similar_documents)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved documents: ['artery pip bisphosphate pip bisphosphate po partial pressure of oxygen pv plasma volume venous pressure correlation coefficient right variable group registration ranking results system rankl receptor activator of nuclear factor ligand rr relative risk respiratory rate rv residual volume right ventricle right ventricular se standard error of the mean siadh syndrome of inappropriate secretion of antidiuretic hormone sv splenic vein stroke volume tca tricarboxylic acid cycle tricyclic antidepressant vasopressin receptors vd volume of distribution vdj variable diversity joining gene segments rearranged to form ig genes vh variable region heavy chain antibody vl variable region light chain antibody vpl ventral posterior nucleus lateral vpm ventral posterior nucleus medial vpn vancomycin polymyxin nystatin media ratio xr xlinked recessive xxxy normal complement of sex chromosomes for femalemale zdv zidovudine formerly azt', 'artery pip bisphosphate pip bisphosphate po 

## Generation Method

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model.to(device)


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

... (7 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3LongRoPEScaledRotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# We pin the exact verified revision hash to address remote code execution security warnings
model_id = "microsoft/Phi-3.5-mini-instruct"
revision_hash = "2fe192450127e6a83f7441aef6e3ca586c338b77"

generation_tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=revision_hash,
    trust_remote_code=True
)

# Loading the model with updated configurations to suppress deprecation warnings:
# 1. 'dtype' is used instead of the deprecated 'torch_dtype'
# 2. 'attn_implementation="eager"' avoids warnings regarding flash-attention requirements
generation_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=revision_hash,
    trust_remote_code=True,
    dtype=torch.float16,
    attn_implementation="eager",
    device_map="auto"
)

/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [21]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# NOTE: If you still get a CUDA error, you MUST restart the notebook runtime
# (Runtime -> Restart session) because `device.reset()` in a previous cell
# permanently broke PyTorch's connection to the GPU for this session.

generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=True,
    torch_dtype=torch.float16,  # Load in 16-bit to prevent Out Of Memory errors
    device_map="auto"           # Automatically moves the model to the GPU safely
)


/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [22]:

# Function to generate a response using retrieved context
def generate_response(query, context, max_new_tokens=100):
    input_text = f"User query: {query}\n\nContext:\n{context}\n\nAnswer:"

    # Tokenize the input and move tensors to GPU
    inputs = generation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to("cuda")

    # Generate response using max_new_tokens to control output length
    with torch.no_grad():
        outputs = generation_model.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            num_return_sequences=1,
            use_cache=False  # Bypass cache incompatibility
        )

    # Decode the generated response
    response_text = generation_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response_text

# Testing generation with retrieved documents as context
retrieved_text = " ".join(similar_documents)  # Concatenate retrieved documents as context
response = generate_response(sample_query, retrieved_text)
print("Generated response:", response)


/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Generated response: User query: What are the symptoms of heart failure?

Context:
artery pip bisphosphate pip bisphosphate po partial pressure of oxygen pv plasma volume venous pressure correlation coefficient right variable group registration ranking results system rankl receptor activator of nuclear factor ligand rr relative risk respiratory rate rv residual volume right ventricle right ventricular se standard error of the mean siadh syndrome of inappropriate secretion of antidiuretic hormone sv splenic vein stroke volume tca tricarboxylic acid cycle tricyclic antidepressant vasopressin receptors vd volume of distribution vdj variable diversity joining gene segments rearranged to form ig genes vh variable region heavy chain antibody vl variable region light chain antibody vpl ventral posterior nucleus lateral vpm ventral posterior nucleus medial vpn vancomycin polymyxin nystatin media ratio xr xlinked recessive xxxy normal complement of sex chromosomes for femalemale zdv zidovudine f